# Daily (from hourly rollup) → BigQuery (thin notebook)

**Git holds only the reusable functions** (the `dhan-pipeline` package).
**This notebook holds every variable** — dates, table names, worker/rate knobs.
Edit the VARIABLES cell, then run.

Builds daily bars by aggregating 60m candles (rate-limited, parallel, retried),
then upserts via the same staging+MERGE path as the main daily pipeline.
Rows land tagged `exchange='TEMP'` so this run's data stays identifiable and
easy to clean up separately.

In [ ]:
# 1. Install the shared functions from GitHub (fast: skips deps Colab already has)
!pip install -q --force-reinstall --no-deps "git+https://github.com/rajatjain1992/dhan-pipeline.git"

In [ ]:
# 2. Auth: BigQuery via Colab, Drive for the service-account JSON (needed for gspread)
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

In [ ]:
# 3. ===== VARIABLES — edit these, this is the only place values live =====
from dhan_pipeline import Config, run_daily_from_hourly
from dhan_pipeline import load_scrip_mapping, gspread_client

START_DATE  = '2026-08-06'   # first window start (YYYY-MM-DD)
N           = 1             # number of windows
STEP_DAYS   = 7             # gap between window starts
WINDOW_DAYS = 1             # length of each window
INTERVAL    = 60            # candle minutes, rolled up to daily

MAX_WORKERS  = 4            # parallel fetch threads
RATE_PER_SEC = 4            # max API calls per second
MAX_RETRIES  = 3

cfg = Config(
    dhan_client_id    = 'PASTE_CLIENT_ID',
    dhan_access_token = 'PASTE_FRESH_DHAN_TOKEN',
    service_account_file = '/content/drive/MyDrive/Colab Notebooks/rajat-trade-c411eaec7c51.json',
    project_id    = 'rajat-trade',
    dataset_id    = 'stock_data_set',
    daily_table   = 'stock_daily_prices_dhan',
    staging_table = 'stock_daily_prices_dhan_staging',
    sheet_key          = '1aoEgOhQkAAv8b2NqAWtZUYXG41rOal77i0XasevyNtE',
    list_worksheet     = 'my_list',
    negative_worksheet = 'Negative List',
)

In [ ]:
# 3b. Load the scrip list from the Google Sheet (or subset it first if you want).
gc = gspread_client(cfg)
scrip_mapping = load_scrip_mapping(cfg, gc)
print(f'Total scrips: {len(scrip_mapping)}')

In [ ]:
# 4. Run: fetch hourly (parallel, rate-limited) -> aggregate to daily -> upsert -> cleanup
result = run_daily_from_hourly(
    cfg, scrip_mapping, START_DATE,
    n=N, step_days=STEP_DAYS, window_days=WINDOW_DAYS, interval=INTERVAL,
    max_workers=MAX_WORKERS, rate_per_sec=RATE_PER_SEC, max_retries=MAX_RETRIES,
)
result